# Watem NB02 — Management scenarios
Start from frozen NB01. Edit the three shapefiles in ArcGIS/QGIS, ZIP them and upload. C/CN adjustments are illustrative; grass kTC proxy is unvalidated and does not alter routing. SOC values document assumptions.

In [ ]:
from google.colab import drive,files
from pathlib import Path
import os, shutil, json, re, zipfile, tempfile
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from rasterio.features import rasterize
from shapely.ops import unary_union
from shapely.geometry import box
from IPython.display import display
drive.mount('/content/drive')
SITE_ID='my_catchment' #@param {type:'string'}
assert SITE_ID and SITE_ID.replace('_','').replace('-','').isalnum()
ROOT=Path('/content/drive/MyDrive/Watem')
SITE=ROOT/'student_sites'/SITE_ID; BASELINE=SITE/'baseline'
assert (BASELINE/'complete.json').exists(),'Complete NB01 first'
import urllib.request, sys
helper_dir=Path('/content/watem_student_helpers'); helper_dir.mkdir(exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/kvanoost/Watem/683decfd6ad9bced8cd0b431a772b167d583291e/watem_student_common.py',helper_dir/'watem_student_common.py')
sys.path.insert(0,str(helper_dir))
import watem_student_common as ws


## 1. Upload and inspect

In [ ]:
SCENARIO_NAME='S01_my_design' #@param {type:'string'}
assert re.fullmatch(r'[A-Za-z0-9_-]{1,60}',SCENARIO_NAME) and SCENARIO_NAME.lower()!='baseline'
DEST=SITE/'scenarios'/SCENARIO_NAME
assert not DEST.exists(),'Use a unique name; never overwrite a scenario'
uploaded=files.upload()
assert len(uploaded)==1
work=Path(tempfile.mkdtemp(prefix='watem_scenario_'))
with zipfile.ZipFile(next(iter(uploaded))) as z:
    for f in z.infolist():
        assert not Path(f.filename).is_absolute() and '..' not in Path(f.filename).parts
    z.extractall(work)
layers={}
for name in ['C_change','CN_change','grass']:
    hits=list(work.rglob(name+'.shp')); assert len(hits)<=1
    if hits:
        g=gpd.read_file(hits[0]); assert g.crs is not None
        g=g.to_crs(31370); assert g.geometry.is_valid.all() and not g.geometry.is_empty.any()
        layers[name]=g
assert layers,'No recognized shapefiles in ZIP'
cfg=ws.read_ini(BASELINE/'annual/modelinput/inifile.ini'); inp=BASELINE/'annual/modelinput'
prc,prof=ws.read_raster(ws.raster_path(cfg,inp,'Parcel filename')); mask=prc!=0; eligible=prc>0; tr=prof['transform']; res=abs(tr.a)
# Fractional footprint via exact polygon/cell intersection; eligible cropland cells only.
def fraction(geom):
    arr=np.zeros(prc.shape,dtype='float32')
    if geom.is_empty: return arr
    minx,miny,maxx,maxy=geom.bounds
    from rasterio.transform import rowcol
    r0,c0=rowcol(tr,minx,maxy); r1,c1=rowcol(tr,maxx,miny)
    for i in range(max(0,r0),min(arr.shape[0],r1+1)):
        for j in range(max(0,c0),min(arr.shape[1],c1+1)):
            if not eligible[i,j]: continue
            x,y=tr*(j,i); arr[i,j]=geom.intersection(box(x,y-res,x+res,y)).area/res**2
    return arr
def zone_adjust(name,field,lo,hi,neutral):
    a=np.full(prc.shape,neutral,dtype='float32'); used=np.zeros(prc.shape,dtype='float32')
    g=layers.get(name)
    if g is None: return a,used
    assert g.geom_type.isin(['Polygon','MultiPolygon']).all()
    for i in range(len(g)):
        for j in range(i):
            assert g.geometry.iloc[i].intersection(g.geometry.iloc[j]).area<1e-6,'Overlapping zones within same layer'
    for _,r in g.iterrows():
        v=float(r[field]); assert np.isfinite(v) and lo<=v<=hi,(field,v)
        f=fraction(r.geometry); a+=f*(v-neutral); used+=f
    assert used.max()<=1.0001
    return a,used
cm,ca=zone_adjust('C_change','c_mult',0,1,1)
cd,na=zone_adjust('CN_change','cn_delta',-20,0,0)
grass=layers.get('grass'); gf=np.zeros(prc.shape,dtype='float32')
if grass is not None and len(grass):
    assert grass.geom_type.isin(['LineString','MultiLineString']).all()
    widths=pd.to_numeric(grass.width_m); assert widths.between(1,40).all()
    gf=fraction(unary_union([g.buffer(w/2,cap_style=2) for g,w in zip(grass.geometry,widths)]))
areas=dict(C_area_ha=float(ca.sum()*res**2/10000),CN_area_ha=float(na.sum()*res**2/10000),grass_area_ha=float(gf.sum()*res**2/10000))
print(areas)
fig,ax=plt.subplots(1,3,figsize=(14,5))
for a,v,t in zip(ax,[cm,cd,gf],['C multiplier','CN change','Grass fraction']):
    im=a.imshow(np.ma.masked_where(~mask,v)); a.set_title(t); fig.colorbar(im,ax=a,shrink=.6)
plt.show()


## 2. Run named independent scenario

In [ ]:
ACCEPT=False #@param {type:'boolean'}
assert ACCEPT,'Inspect intervention maps and areas first'
# Baseline frozen; each scenario starts from its original native files.
binary=Path('/content/watem_sedem')
shutil.copy2(ROOT/'software/watem_sedem_5.0.3',binary); binary.chmod(0o755)
DEST.mkdir(parents=True)
shutil.copy2(next(iter(uploaded)),DEST/'design.zip')
(DEST/'design.json').write_text(json.dumps(dict(name=SCENARIO_NAME,**areas,grass_method='fractional C; event-equivalent CN; provisional harmonic kTC with grass kTC=5; PRC/routing unchanged',soil_status='SOC fields descriptive only'),indent=2))
rows=[]
for mode in ['annual','event_20mm','event_40mm','event_60mm']:
    rain=None if mode=='annual' else int(mode.split('_')[1][:-2])
    dest=DEST/mode
    cfg,inp,out=ws.clone_run(BASELINE/mode,dest,binary,rain)
    cp=ws.raster_path(cfg,inp,'C factor map filename'); c,p=ws.read_raster(cp)
    cnew=c.copy(); cnew[eligible]=((c*cm)*(1-gf)+.01*gf)[eligible]
    assert np.isfinite(cnew[mask]).all() and np.all((cnew[mask]>=0)&(cnew[mask]<=1))
    ws.native_write(cp,cnew,p)
    # Illustrative subpixel trapping proxy; not calibrated width-aware routing.
    kp=ws.raster_path(cfg,inp,'ktc map filename'); k,p=ws.read_raster(kp)
    use=eligible & (gf>0)
    k[use]=1/((1-gf[use])/np.maximum(k[use],1e-6)+gf[use]/5.)
    ws.native_write(kp,k,p)
    if rain is not None:
        npth=ws.raster_path(cfg,inp,'CN map filename'); cn,p=ws.read_raster(npth)
        cnnew=cn.copy(); cnnew[eligible]=np.clip(cn[eligible]+cd[eligible],30,100)
        def excess(c):
            S=25400/np.clip(c,1,100)-254; return np.maximum(rain-.2*S,0)**2/np.maximum(rain+.8*S,1e-9)
        # Preserve baseline group B assumption; grass CN=61. Invert event excess.
        target=(1-gf)*excess(cnnew)+gf*excess(np.full(cn.shape,61.))
        low=np.full(cn.shape,1.); high=np.full(cn.shape,100.)
        for i in range(32):
            mid=(low+high)/2; gt=excess(mid)>target; high=np.where(gt,mid,high); low=np.where(gt,low,mid)
        cnnew[use & (target>0)]=((low+high)/2)[use & (target>0)]
        cnnew[use & (target<=0)]=np.minimum(cnnew,61)[use & (target<=0)]
        assert np.all((cnnew[mask]>0)&(cnnew[mask]<=100)); ws.native_write(npth,cnnew,p)
    ws.execute(dest,binary)
    row=ws.report(dest,mode,rain); row.update(areas); rows.append(row)
    ws.tif_write(dest/'report/grass_fraction.tif',gf,prof,mask)
    base,_=ws.read_raster(BASELINE/mode/'report/erosion_t_ha.tif'); cur,_=ws.read_raster(dest/'report/erosion_t_ha.tif')
    ws.tif_write(dest/'report/erosion_difference.tif',cur-base,prof,mask & (cur!=-9999)&(base!=-9999))
    if rain is not None:
        base,_=ws.read_raster(BASELINE/mode/'report/local_runoff_mm.tif'); cur,_=ws.read_raster(dest/'report/local_runoff_mm.tif')
        ws.tif_write(dest/'report/runoff_difference.tif',cur-base,prof,mask & (cur!=-9999)&(base!=-9999))
pd.DataFrame(rows).to_csv(DEST/'summary.csv',index=False); display(pd.DataFrame(rows))
(DEST/'complete.json').write_text(json.dumps(dict(site=SITE_ID,scenario=SCENARIO_NAME,**areas),indent=2))
export=work/'results'; export.mkdir()
for mode in ['annual','event_20mm','event_40mm','event_60mm']: shutil.copytree(DEST/mode/'report',export/mode)
for name in ['design.zip','design.json','summary.csv']: shutil.copy2(DEST/name,export/name)
files.download(shutil.make_archive(str(DEST/'results'),'zip',export))
